# Classification de prunes africaines — pipeline d'entraînement (Colab)

Ce notebook exécute le pipeline de bout en bout et **écrit les métriques dans
`results/`** pour qu'elles puissent être versionnées. Sans cette dernière étape,
un entraînement ne laisse aucune trace vérifiable.

Prérequis : un runtime GPU (`Exécution > Modifier le type d'exécution > GPU`)
et un token Kaggle (`kaggle.json`).

In [ ]:
# 1. Dépendances
!pip install -q torch torchvision pytorch-lightning albumentations timm kaggle
!pip install -q pandas matplotlib seaborn scikit-learn pillow torchmetrics onnx

import torch
print(f"PyTorch      : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
else:
    print("ATTENTION : pas de GPU, l'entraînement sera inutilisable en pratique.")

In [ ]:
# 2. Récupérer le code du dépôt
# Les modules du pipeline vivent ici ; sans ce clone les imports de la cellule 4 échouent.
import os, sys

REPO = "https://github.com/CodeStorm-mbe/african-plums-classifier.git"
if not os.path.isdir("african-plums-classifier"):
    !git clone -q $REPO
sys.path.insert(0, "/content/african-plums-classifier")
print(os.listdir("/content/african-plums-classifier"))

In [ ]:
# 3. Token Kaggle — nécessaire pour télécharger le dataset
# Dépose kaggle.json (Kaggle > Settings > Create New Token) via le sélecteur ci-dessous.
from google.colab import files
import os, shutil

if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Token Kaggle en place.")

In [ ]:
# 4. Exécution du pipeline
from training_pipeline_enhanced import EnhancedPlumClassificationPipeline

pipeline = EnhancedPlumClassificationPipeline(
    base_dir='/content/plum_classifier',
    kaggle_dataset='arnaudfadja/african-plums-quality-and-defect-assessment-data',
    use_wandb=False,
    use_cross_validation=False,   # passer à True (n_folds=5) pour le run complet
    use_tta=True,
    use_ensemble=False,
)

results = pipeline.run_pipeline()
print(f"Modèle      : {results['model_path']}")
print(f"Export ONNX : {results['onnx_path']}")

## Capturer les résultats

L'étape qui manquait. Le pipeline calcule déjà exactitude, F1 et matrice de
confusion ; ils vivaient uniquement dans les logs du runtime et disparaissaient
avec la session. On les écrit sur disque, puis on les télécharge pour les
committer dans `results/`.

In [ ]:
# 5. Persister les métriques
import json, os, shutil, numpy as np

RESULTS_DIR = "/content/african-plums-classifier/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

ev = results['training_results']['evaluation_results']

metrics = {
    "accuracy":  float(ev['accuracy']),
    "precision": float(ev['precision']),
    "recall":    float(ev['recall']),
    "f1":        float(ev['f1']),
    "confusion_matrix": np.asarray(ev['confusion_matrix']).tolist(),
    "classification_report": ev['classification_report'],
    "n_classes": 6,
    "cross_validation": pipeline.use_cross_validation,
    "tta": pipeline.use_tta,
    "ensemble": pipeline.use_ensemble,
}

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

cm_src = ev.get('confusion_matrix_path')
if cm_src and os.path.exists(cm_src):
    shutil.copy(cm_src, os.path.join(RESULTS_DIR, "confusion_matrix.png"))

print(f"Exactitude (test) : {metrics['accuracy']:.4f}")
print(f"F1 (macro)        : {metrics['f1']:.4f}")
print(f"Écrit dans        : {RESULTS_DIR}")

In [ ]:
# 6. Récupérer les artefacts pour les versionner
from google.colab import files
import shutil

shutil.make_archive("/content/results", "zip", RESULTS_DIR)
files.download("/content/results.zip")